# 3D Medical Image Registration & Segmentation

**Welcome!** This notebook demonstrates the core concepts behind 3D medical image
registration and segmentation using **synthetic data** and pure Python libraries
(NumPy, SciPy, Matplotlib).

## What is Medical Image Registration?

Medical image registration aligns two or more images into a common coordinate system.
This is essential when:

- Comparing brain scans taken at different times (longitudinal studies)
- Fusing information from different imaging modalities (T1, T2, FLAIR)
- Mapping patient anatomy onto a standard atlas (e.g., MNI152)

## What is Image Segmentation?

Segmentation partitions an image into meaningful regions, for example separating
brain tissue from skull and background, or identifying tumour regions.

## What We Will Do

1. **Create** synthetic 3D brain-like volumes with known ground truth
2. **Preprocess** volumes using our utility functions (normalize, mask, fill holes)
3. **Register** a misaligned volume back to a reference using rigid transforms
4. **Segment** structures using thresholding and mask operations
5. **Evaluate** registration and segmentation quality with Dice and IoU metrics
6. **Visualize** 2D slices through the 3D volumes at each stage

All image processing utilities are imported from `src/image_utils.py`, which
contains functions extracted from the project's ANTsPy-based notebooks.

---
## Part 1: Environment Setup

We configure matplotlib for headless (non-interactive) rendering and import
the required libraries.

In [ ]:
# Configure matplotlib for headless rendering (must come before other imports)
import matplotlib
matplotlib.use('Agg')

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.ndimage import affine_transform, gaussian_filter

# Set working directory
os.chdir(os.path.dirname(os.path.abspath('__file__')) if os.path.exists('__file__') else os.getcwd())
sys.path.insert(0, os.getcwd())

# Import our utility functions from src/image_utils.py
from src.image_utils import (
    normalize_volume,
    compute_binary_mask,
    fill_holes_3d,
    apply_binary_mask,
    compute_volume_statistics,
    compute_neighbor_structure,
    threshold_volume,
    compute_bounding_box,
    extract_brain_roi,
    compute_center_of_mass,
    compute_dice_coefficient,
    compute_volume_overlap,
    clip_volume,
    pad_volume_to_shape,
    crop_volume_to_bounding_box,
    resample_volume_nearest,
)

# Reproducibility
np.random.seed(42)

print(f"Working directory: {os.getcwd()}")
print(f"NumPy version:     {np.__version__}")
print(f"SciPy version:     {ndimage.__name__} (from scipy {__import__('scipy').__version__})")
print("\nAll libraries and utilities loaded successfully.")

---
## Part 2: Creating Synthetic 3D Brain-Like Volumes

Real brain MRI data requires heavy dependencies (nibabel, ANTsPy) to load.
Instead, we create **synthetic 3D volumes** that mimic the key properties of
brain imaging data:

- An ellipsoidal "brain" region inside a larger volume
- Internal structures (grey matter, white matter, ventricles) as Gaussian blobs
- Different intensity profiles for different "modalities" (T1-like, T2-like)
- Realistic noise levels

This gives us known ground truth for evaluating our algorithms.

In [ ]:
def create_synthetic_brain(shape=(64, 64, 64), modality='T1', noise_level=0.05):
    """
    Create a synthetic 3D brain-like volume.
    
    Parameters
    ----------
    shape : tuple
        Volume dimensions (D, H, W).
    modality : str
        'T1' (white matter bright) or 'T2' (CSF bright).
    noise_level : float
        Standard deviation of additive Gaussian noise.
    
    Returns
    -------
    volume : np.ndarray
        The synthetic volume.
    brain_mask : np.ndarray
        Binary mask of the brain region.
    structures : dict
        Dictionary of named structure masks (ground truth segmentation).
    """
    D, H, W = shape
    volume = np.zeros(shape, dtype=np.float64)
    
    # Coordinate grids centered at volume midpoint
    zz, yy, xx = np.mgrid[0:D, 0:H, 0:W]
    cz, cy, cx = D // 2, H // 2, W // 2
    
    # --- Brain shell (ellipsoid) ---
    brain_ellipsoid = (
        ((zz - cz) / (D * 0.38)) ** 2 +
        ((yy - cy) / (H * 0.42)) ** 2 +
        ((xx - cx) / (W * 0.35)) ** 2
    )
    brain_mask = (brain_ellipsoid < 1.0).astype(np.float32)
    
    # --- Internal structures ---
    structures = {}
    
    # White matter core (smaller ellipsoid)
    wm_ellipsoid = (
        ((zz - cz) / (D * 0.25)) ** 2 +
        ((yy - cy) / (H * 0.28)) ** 2 +
        ((xx - cx) / (W * 0.22)) ** 2
    )
    wm_mask = (wm_ellipsoid < 1.0).astype(np.float32)
    structures['white_matter'] = wm_mask
    
    # Grey matter = brain minus white matter
    gm_mask = brain_mask * (1.0 - wm_mask)
    structures['grey_matter'] = (gm_mask > 0.5).astype(np.float32)
    
    # Ventricles (two small blobs)
    vent_left = np.exp(-0.5 * (
        ((zz - cz) / 4) ** 2 +
        ((yy - cy) / 6) ** 2 +
        ((xx - (cx - 6)) / 3) ** 2
    ))
    vent_right = np.exp(-0.5 * (
        ((zz - cz) / 4) ** 2 +
        ((yy - cy) / 6) ** 2 +
        ((xx - (cx + 6)) / 3) ** 2
    ))
    vent_mask = ((vent_left > 0.3) | (vent_right > 0.3)).astype(np.float32)
    structures['ventricles'] = vent_mask
    
    # Tumour-like lesion (small blob in left hemisphere)
    tumour = np.exp(-0.5 * (
        ((zz - (cz + 5)) / 3) ** 2 +
        ((yy - (cy - 8)) / 4) ** 2 +
        ((xx - (cx - 10)) / 3) ** 2
    ))
    tumour_mask = (tumour > 0.3).astype(np.float32)
    structures['lesion'] = tumour_mask
    
    # --- Assign intensities based on modality ---
    if modality == 'T1':
        # T1: WM bright, GM medium, CSF/ventricles dark
        volume += gm_mask * 0.6
        volume += wm_mask * 0.9
        volume += vent_mask * 0.15
        volume += tumour_mask * 0.75
    else:  # T2
        # T2: CSF/ventricles bright, GM medium, WM darker
        volume += gm_mask * 0.55
        volume += wm_mask * 0.35
        volume += vent_mask * 0.95
        volume += tumour_mask * 0.7
    
    # Apply brain mask (zero outside brain)
    volume = volume * brain_mask
    
    # Smooth to make it more realistic
    volume = gaussian_filter(volume, sigma=1.0)
    volume = volume * brain_mask  # re-apply mask after smoothing
    
    # Add noise
    noise = np.random.normal(0, noise_level, shape)
    volume = volume + noise * brain_mask  # noise only inside brain
    volume = np.clip(volume, 0, 1)
    
    return volume, brain_mask, structures


# Create two modalities of the same synthetic brain
t1_volume, brain_mask_gt, structures_gt = create_synthetic_brain(modality='T1')
t2_volume, _, _ = create_synthetic_brain(modality='T2')

print(f"Volume shape:     {t1_volume.shape}")
print(f"Brain mask voxels: {int(brain_mask_gt.sum())} / {brain_mask_gt.size} "
      f"({100 * brain_mask_gt.sum() / brain_mask_gt.size:.1f}%)")
print(f"\nGround-truth structures:")
for name, mask in structures_gt.items():
    print(f"  {name:15s}: {int(mask.sum()):6d} voxels")

In [ ]:
# Visualize the synthetic brain: axial, coronal, sagittal slices
def show_volume_slices(volume, title='', cmap='gray', slice_indices=None):
    """Display three orthogonal slices through a 3D volume."""
    D, H, W = volume.shape
    if slice_indices is None:
        slice_indices = (D // 2, H // 2, W // 2)
    sz, sy, sx = slice_indices
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    axes[0].imshow(volume[sz, :, :], cmap=cmap, origin='lower', vmin=0, vmax=1)
    axes[0].set_title(f'Axial (z={sz})')
    axes[0].set_xlabel('Left-Right')
    axes[0].set_ylabel('Anterior-Posterior')
    
    axes[1].imshow(volume[:, sy, :], cmap=cmap, origin='lower', vmin=0, vmax=1)
    axes[1].set_title(f'Coronal (y={sy})')
    axes[1].set_xlabel('Left-Right')
    axes[1].set_ylabel('Superior-Inferior')
    
    axes[2].imshow(volume[:, :, sx], cmap=cmap, origin='lower', vmin=0, vmax=1)
    axes[2].set_title(f'Sagittal (x={sx})')
    axes[2].set_xlabel('Anterior-Posterior')
    axes[2].set_ylabel('Superior-Inferior')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig


fig = show_volume_slices(t1_volume, title='Synthetic Brain - T1-weighted')
fig.savefig('synthetic_brain_t1.png', dpi=100, bbox_inches='tight')
plt.show()

fig = show_volume_slices(t2_volume, title='Synthetic Brain - T2-weighted')
fig.savefig('synthetic_brain_t2.png', dpi=100, bbox_inches='tight')
plt.show()

print("T1: white matter appears bright, ventricles dark")
print("T2: ventricles appear bright, white matter darker")
print("Both images show the same anatomy with different contrast.")

---
## Part 3: Image Preprocessing

Before registration or segmentation, medical images must be preprocessed:

1. **Intensity normalization** - scale values to a standard range
2. **Brain mask computation** - identify the brain region
3. **Hole filling** - clean up mask artifacts
4. **ROI extraction** - isolate the brain from background

We use the functions from `src/image_utils.py` which were extracted from the
project's ANTsPy-based notebooks.

In [ ]:
# Step 1: Normalize the T1 volume to [0, 1]
t1_normalized = normalize_volume(t1_volume, new_min=0.0, new_max=1.0)
stats_raw = compute_volume_statistics(t1_volume)
stats_norm = compute_volume_statistics(t1_normalized)

print("T1 Volume Statistics")
print(f"  Before normalization: min={stats_raw['min']:.4f}, max={stats_raw['max']:.4f}, "
      f"mean={stats_raw['mean']:.4f}")
print(f"  After normalization:  min={stats_norm['min']:.4f}, max={stats_norm['max']:.4f}, "
      f"mean={stats_norm['mean']:.4f}")

# Step 2: Compute brain mask from the normalized volume
# Using a threshold slightly above zero catches the brain region
computed_mask = compute_binary_mask(t1_normalized, threshold=0.05)
print(f"\nComputed brain mask: {int(computed_mask.sum())} voxels "
      f"(ground truth: {int(brain_mask_gt.sum())})")

# Step 3: Fill holes in the computed mask
structure = compute_neighbor_structure(size=3)
filled_mask = fill_holes_3d(computed_mask, structure=structure)
print(f"After hole-filling:  {int(filled_mask.sum())} voxels")

# Step 4: Apply mask to extract brain ROI
brain_roi = apply_binary_mask(t1_normalized, filled_mask)
roi_stats = compute_volume_statistics(brain_roi)
print(f"\nBrain ROI: {roi_stats['nonzero_count']} non-zero voxels")

# Compute Dice overlap between computed mask and ground truth
mask_dice = compute_dice_coefficient(filled_mask, brain_mask_gt)
mask_iou = compute_volume_overlap(filled_mask, brain_mask_gt)
print(f"\nMask quality vs ground truth:")
print(f"  Dice coefficient: {mask_dice:.4f}")
print(f"  IoU (Jaccard):    {mask_iou:.4f}")

In [ ]:
# Visualize preprocessing steps
mid_z = t1_volume.shape[0] // 2

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

axes[0, 0].imshow(t1_volume[mid_z], cmap='gray', origin='lower')
axes[0, 0].set_title('1. Raw T1 Volume')

axes[0, 1].imshow(t1_normalized[mid_z], cmap='gray', origin='lower')
axes[0, 1].set_title('2. Normalized [0, 1]')

axes[0, 2].imshow(computed_mask[mid_z], cmap='gray', origin='lower')
axes[0, 2].set_title('3. Computed Mask')

axes[1, 0].imshow(filled_mask[mid_z], cmap='gray', origin='lower')
axes[1, 0].set_title('4. Hole-Filled Mask')

axes[1, 1].imshow(brain_roi[mid_z], cmap='gray', origin='lower')
axes[1, 1].set_title('5. Brain ROI')

# Overlay: mask boundary on original
from scipy.ndimage import binary_erosion
boundary = filled_mask[mid_z] - binary_erosion(filled_mask[mid_z]).astype(np.float32)
axes[1, 2].imshow(t1_normalized[mid_z], cmap='gray', origin='lower')
axes[1, 2].imshow(boundary, cmap='Reds', alpha=0.6, origin='lower')
axes[1, 2].set_title('6. Mask Boundary Overlay')

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Preprocessing Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('preprocessing_pipeline.png', dpi=100, bbox_inches='tight')
plt.show()

print("The preprocessing pipeline normalizes intensities, computes a brain mask,")
print("fills holes, and extracts the brain region of interest.")

---
## Part 4: Image Registration

**Registration** aligns a *moving* image to a *fixed* (reference) image.

We demonstrate **rigid registration** using `scipy.ndimage.affine_transform`:
- Create a misaligned version of the brain (translation + rotation)
- Apply the known inverse transform to recover alignment
- Evaluate quality using voxel correlation and mask overlap

In a real pipeline, **ANTsPy** would compute the optimal transform automatically.
Here we show the concept with a known ground-truth transform.

In [ ]:
# Create a misaligned ("moving") volume by applying a known rigid transform

def apply_rigid_transform(volume, rotation_deg=0, translation=(0, 0, 0), axis='z'):
    """
    Apply a rigid transform (rotation + translation) to a 3D volume.
    
    Parameters
    ----------
    volume : np.ndarray
        3D input volume.
    rotation_deg : float
        Rotation angle in degrees.
    translation : tuple
        (tz, ty, tx) translation in voxels.
    axis : str
        Rotation axis: 'x', 'y', or 'z'.
    
    Returns
    -------
    transformed : np.ndarray
        Transformed volume.
    matrix : np.ndarray
        3x3 rotation matrix used.
    offset : np.ndarray
        Translation offset used.
    """
    theta = np.deg2rad(rotation_deg)
    c, s = np.cos(theta), np.sin(theta)
    
    if axis == 'z':
        rot = np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
    elif axis == 'y':
        rot = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    else:  # x
        rot = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    
    # Rotate around volume center
    center = np.array(volume.shape) / 2.0
    offset = center - rot @ center + np.array(translation)
    
    transformed = affine_transform(volume, rot, offset=offset, order=1, mode='constant', cval=0.0)
    return transformed, rot, offset


# Apply a known misalignment: 8-degree rotation + (3, -2, 4) voxel shift
ROTATION_DEG = 8.0
TRANSLATION = (3.0, -2.0, 4.0)

moving_volume, rot_matrix, offset_used = apply_rigid_transform(
    t1_normalized,
    rotation_deg=ROTATION_DEG,
    translation=TRANSLATION,
    axis='z'
)

# Also transform the brain mask (for evaluation)
moving_mask, _, _ = apply_rigid_transform(
    brain_mask_gt,
    rotation_deg=ROTATION_DEG,
    translation=TRANSLATION,
    axis='z'
)
moving_mask = (moving_mask > 0.5).astype(np.float32)

# Measure misalignment
misalign_dice = compute_dice_coefficient(moving_mask, brain_mask_gt)
print(f"Applied misalignment: {ROTATION_DEG} deg rotation + {TRANSLATION} voxel shift")
print(f"Mask Dice before registration: {misalign_dice:.4f} (1.0 = perfect alignment)")

In [ ]:
# Register the moving volume back to the fixed volume
# Since we know the forward transform, we apply the inverse

# Inverse of rotation matrix is its transpose
inv_rot = rot_matrix.T
# Inverse offset
center = np.array(t1_normalized.shape) / 2.0
inv_offset = center - inv_rot @ center - inv_rot @ np.array(TRANSLATION)

# Apply inverse transform to recover alignment
registered_volume = affine_transform(
    moving_volume, inv_rot, offset=inv_offset, order=1, mode='constant', cval=0.0
)
registered_mask_raw = affine_transform(
    moving_mask, inv_rot, offset=inv_offset, order=0, mode='constant', cval=0.0
)
registered_mask = (registered_mask_raw > 0.5).astype(np.float32)

# Evaluate registration quality
reg_dice = compute_dice_coefficient(registered_mask, brain_mask_gt)
reg_iou = compute_volume_overlap(registered_mask, brain_mask_gt)

# Voxel-wise correlation within brain region
brain_voxels_fixed = t1_normalized[brain_mask_gt > 0]
brain_voxels_registered = registered_volume[brain_mask_gt > 0]
correlation = np.corrcoef(brain_voxels_fixed, brain_voxels_registered)[0, 1]

print("Registration Results:")
print("=" * 50)
print(f"  Mask Dice (before):  {misalign_dice:.4f}")
print(f"  Mask Dice (after):   {reg_dice:.4f}")
print(f"  Mask IoU (after):    {reg_iou:.4f}")
print(f"  Voxel correlation:   {correlation:.4f}")
print(f"\nImprovement: {reg_dice - misalign_dice:+.4f} Dice")

In [ ]:
# Visualize registration: before and after
mid_z = t1_volume.shape[0] // 2

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Top row: images
axes[0, 0].imshow(t1_normalized[mid_z], cmap='gray', origin='lower')
axes[0, 0].set_title('Fixed (Reference)')

axes[0, 1].imshow(moving_volume[mid_z], cmap='gray', origin='lower')
axes[0, 1].set_title(f'Moving (misaligned {ROTATION_DEG}deg)')

axes[0, 2].imshow(registered_volume[mid_z], cmap='gray', origin='lower')
axes[0, 2].set_title('Registered (corrected)')

# Bottom row: difference maps
diff_before = np.abs(t1_normalized[mid_z] - moving_volume[mid_z])
diff_after = np.abs(t1_normalized[mid_z] - registered_volume[mid_z])

axes[1, 0].imshow(diff_before, cmap='hot', origin='lower', vmin=0, vmax=0.5)
axes[1, 0].set_title(f'|Fixed - Moving|\n(Dice={misalign_dice:.3f})')

axes[1, 1].imshow(diff_after, cmap='hot', origin='lower', vmin=0, vmax=0.5)
axes[1, 1].set_title(f'|Fixed - Registered|\n(Dice={reg_dice:.3f})')

# Checkerboard overlay
checker_size = 8
checker = np.zeros_like(t1_normalized[mid_z])
for i in range(0, checker.shape[0], checker_size):
    for j in range(0, checker.shape[1], checker_size):
        if ((i // checker_size) + (j // checker_size)) % 2 == 0:
            checker[i:i+checker_size, j:j+checker_size] = t1_normalized[mid_z, i:i+checker_size, j:j+checker_size]
        else:
            checker[i:i+checker_size, j:j+checker_size] = registered_volume[mid_z, i:i+checker_size, j:j+checker_size]

axes[1, 2].imshow(checker, cmap='gray', origin='lower')
axes[1, 2].set_title('Checkerboard\n(Fixed vs Registered)')

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Image Registration: Before vs After', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('registration_results.png', dpi=100, bbox_inches='tight')
plt.show()

print("Top row: the moving image is visibly misaligned; after registration it matches.")
print("Bottom row: difference maps show the error is greatly reduced.")
print("Checkerboard view alternates tiles from fixed and registered to check alignment.")

---
## Part 5: Image Segmentation

**Segmentation** partitions the image into anatomically meaningful regions.

We demonstrate:
1. **Intensity thresholding** - separate tissue types by intensity
2. **Morphological operations** - clean up segmentation masks
3. **Multi-class segmentation** - identify WM, GM, ventricles simultaneously

In practice, deep learning models (e.g., MONAI U-Net) are used for accurate
segmentation. Here we show the fundamental concepts these models build upon.

In [ ]:
# Segment the T1 volume using intensity thresholds
# Since our synthetic T1 has known intensities, we can set thresholds

# Work with the brain ROI (background already removed)
brain_only = extract_brain_roi(t1_normalized, brain_mask_gt)

# Threshold-based segmentation
# T1 intensities: ventricles ~0.15, GM ~0.6, WM ~0.9
ventricle_seg = threshold_volume(brain_only, threshold=0.0) * (1.0 - threshold_volume(brain_only, threshold=0.25))
ventricle_seg = ventricle_seg * brain_mask_gt  # ensure inside brain

gm_seg = threshold_volume(brain_only, threshold=0.25) * (1.0 - threshold_volume(brain_only, threshold=0.70))

wm_seg = threshold_volume(brain_only, threshold=0.70)

# Clean up with morphological operations
ventricle_seg_clean = fill_holes_3d(ventricle_seg)
ventricle_seg_clean = ventricle_seg_clean * brain_mask_gt

# Evaluate segmentation against ground truth
print("Segmentation Results (Dice Coefficient):")
print("=" * 50)

seg_results = {}
for name, pred, gt in [
    ('White Matter', wm_seg, structures_gt['white_matter']),
    ('Grey Matter', gm_seg, structures_gt['grey_matter']),
    ('Ventricles', ventricle_seg_clean, structures_gt['ventricles']),
]:
    dice = compute_dice_coefficient(pred, gt)
    iou = compute_volume_overlap(pred, gt)
    seg_results[name] = {'dice': dice, 'iou': iou}
    print(f"  {name:15s}: Dice={dice:.4f}, IoU={iou:.4f}")

# Center of mass comparison
print(f"\nCenter of Mass (predicted vs ground truth):")
for name, pred, gt in [
    ('White Matter', wm_seg, structures_gt['white_matter']),
    ('Ventricles', ventricle_seg_clean, structures_gt['ventricles']),
]:
    com_pred = compute_center_of_mass(pred)
    com_gt = compute_center_of_mass(gt)
    if com_pred is not None and com_gt is not None:
        dist = np.sqrt(sum((p - g) ** 2 for p, g in zip(com_pred, com_gt)))
        print(f"  {name:15s}: distance = {dist:.2f} voxels")

In [ ]:
# Visualize segmentation results
mid_z = t1_volume.shape[0] // 2

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Top row: ground truth
axes[0, 0].imshow(t1_normalized[mid_z], cmap='gray', origin='lower')
axes[0, 0].set_title('T1 Volume')

axes[0, 1].imshow(structures_gt['white_matter'][mid_z], cmap='Blues', origin='lower')
axes[0, 1].set_title('GT: White Matter')

axes[0, 2].imshow(structures_gt['grey_matter'][mid_z], cmap='Greens', origin='lower')
axes[0, 2].set_title('GT: Grey Matter')

axes[0, 3].imshow(structures_gt['ventricles'][mid_z], cmap='Reds', origin='lower')
axes[0, 3].set_title('GT: Ventricles')

# Bottom row: our segmentation
# Create composite label map
label_map = np.zeros_like(t1_normalized[mid_z])
label_map[wm_seg[mid_z] > 0] = 1
label_map[gm_seg[mid_z] > 0] = 2
label_map[ventricle_seg_clean[mid_z] > 0] = 3

axes[1, 0].imshow(label_map, cmap='Set1', origin='lower', vmin=0, vmax=4)
axes[1, 0].set_title('All Segments')

axes[1, 1].imshow(wm_seg[mid_z], cmap='Blues', origin='lower')
d = seg_results['White Matter']['dice']
axes[1, 1].set_title(f'Pred: WM (Dice={d:.3f})')

axes[1, 2].imshow(gm_seg[mid_z], cmap='Greens', origin='lower')
d = seg_results['Grey Matter']['dice']
axes[1, 2].set_title(f'Pred: GM (Dice={d:.3f})')

axes[1, 3].imshow(ventricle_seg_clean[mid_z], cmap='Reds', origin='lower')
d = seg_results['Ventricles']['dice']
axes[1, 3].set_title(f'Pred: Vent (Dice={d:.3f})')

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Segmentation: Ground Truth (top) vs Predicted (bottom)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('segmentation_results.png', dpi=100, bbox_inches='tight')
plt.show()

print("Top row: ground-truth structure masks from synthetic data generation.")
print("Bottom row: our threshold-based segmentation with Dice scores.")

---
## Part 6: Combined Visualization

Finally, let's create a comprehensive visualization showing the complete pipeline
from raw data through preprocessing, registration, and segmentation.

In [ ]:
# Comprehensive summary figure
mid_z = t1_volume.shape[0] // 2

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Row 1: Registration pipeline
axes[0, 0].imshow(t1_normalized[mid_z], cmap='gray', origin='lower')
axes[0, 0].set_title('Fixed Volume', fontsize=11)

axes[0, 1].imshow(moving_volume[mid_z], cmap='gray', origin='lower')
axes[0, 1].set_title('Moving (misaligned)', fontsize=11)

axes[0, 2].imshow(registered_volume[mid_z], cmap='gray', origin='lower')
axes[0, 2].set_title(f'Registered (Dice={reg_dice:.3f})', fontsize=11)

diff_after = np.abs(t1_normalized[mid_z] - registered_volume[mid_z])
axes[0, 3].imshow(diff_after, cmap='hot', origin='lower', vmin=0, vmax=0.3)
axes[0, 3].set_title('Registration Error', fontsize=11)

# Row 2: Segmentation
axes[1, 0].imshow(brain_roi[mid_z], cmap='gray', origin='lower')
axes[1, 0].set_title('Brain ROI', fontsize=11)

# Composite segmentation overlay
rgb_overlay = np.stack([t1_normalized[mid_z]] * 3, axis=-1)
rgb_overlay = np.clip(rgb_overlay, 0, 1)
alpha = 0.4
wm_slice = wm_seg[mid_z] > 0
gm_slice = gm_seg[mid_z] > 0
vent_slice = ventricle_seg_clean[mid_z] > 0

rgb_overlay[wm_slice] = (1 - alpha) * rgb_overlay[wm_slice] + alpha * np.array([0.2, 0.4, 1.0])
rgb_overlay[gm_slice] = (1 - alpha) * rgb_overlay[gm_slice] + alpha * np.array([0.2, 0.8, 0.2])
rgb_overlay[vent_slice] = (1 - alpha) * rgb_overlay[vent_slice] + alpha * np.array([1.0, 0.2, 0.2])
rgb_overlay = np.clip(rgb_overlay, 0, 1)

axes[1, 1].imshow(rgb_overlay, origin='lower')
axes[1, 1].set_title('Segmentation Overlay', fontsize=11)

# Bounding box visualization
lesion_mask = structures_gt['lesion']
bbox = compute_bounding_box(lesion_mask)
axes[1, 2].imshow(t1_normalized[mid_z], cmap='gray', origin='lower')
if bbox is not None:
    (z0, z1), (y0, y1), (x0, x1) = bbox
    if z0 <= mid_z <= z1:
        rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                             linewidth=2, edgecolor='yellow', facecolor='none')
        axes[1, 2].add_patch(rect)
axes[1, 2].set_title('Lesion Bounding Box', fontsize=11)

# Metrics summary
axes[1, 3].axis('off')
summary_text = (
    "Pipeline Summary\n"
    "" + "=" * 25 + "\n\n"
    f"Registration:\n"
    f"  Dice: {reg_dice:.4f}\n"
    f"  Correlation: {correlation:.4f}\n\n"
    f"Segmentation (Dice):\n"
    f"  WM:   {seg_results['White Matter']['dice']:.4f}\n"
    f"  GM:   {seg_results['Grey Matter']['dice']:.4f}\n"
    f"  Vent: {seg_results['Ventricles']['dice']:.4f}\n\n"
    f"Volume: {t1_volume.shape}\n"
    f"Brain voxels: {int(brain_mask_gt.sum())}"
)
axes[1, 3].text(0.1, 0.9, summary_text, transform=axes[1, 3].transAxes,
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

for ax in axes.flat:
    if ax != axes[1, 3]:
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('3D Medical Image Registration & Segmentation Pipeline',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('pipeline_summary.png', dpi=100, bbox_inches='tight')
plt.show()

print("Complete pipeline demonstrated: preprocessing, registration, segmentation.")

---
## Part 7: Summary and Real-World Extensions

### What We Demonstrated

This notebook showed the **fundamental concepts** of 3D medical image processing
using synthetic data and basic NumPy/SciPy operations:

| Step | Method Used | Real-World Tool |
|------|-----------|------------------|
| Brain extraction | Intensity thresholding | HD-BET (deep learning) |
| Registration | Known rigid transform | ANTsPy SyN (deformable) |
| Segmentation | Multi-threshold | MONAI U-Net (deep learning) |
| Evaluation | Dice, IoU | Same metrics at higher accuracy |

### How ANTsPy / MONAI Would Improve These Results

**ANTsPy** (used in the project's main notebooks):
- **Deformable registration** via SyN algorithm handles non-rigid anatomical
  differences, not just rigid misalignment
- **Automatic atlas registration** maps patient data to MNI152 standard space
- **Multi-resolution optimization** finds globally optimal transforms
- Typical registration Dice > 0.95 on real brain data

**MONAI** (Medical Open Network for AI):
- **Deep learning segmentation** with U-Net, SegResNet, UNETR architectures
- Trained on BraTS dataset with thousands of annotated brain tumour cases
- Typical tumour segmentation Dice > 0.85 on unseen patients
- GPU-accelerated for processing hundreds of volumes

**HD-BET** (used in the project's atlas processing notebook):
- Deep learning brain extraction that works robustly across scan types
- Far more accurate than intensity thresholding, especially near skull base

### Key Insight

The utility functions in `src/image_utils.py` (normalize, mask, threshold, Dice)
are the **same building blocks** used inside these advanced tools. Understanding
them helps interpret what ANTsPy and MONAI do internally.

### Resources

- **Project notebooks**: `notebooks/3D-image-registration.ipynb` (ANTsPy pipeline)
- **BraTS Challenge**: https://www.synapse.org/brats
- **ANTsPy**: https://github.com/ANTsX/ANTsPy
- **MONAI**: https://monai.io/

In [ ]:
# Final summary
import glob

print("Generated output files:")
for f in sorted(glob.glob('*.png')):
    print(f"  {f}")

print(f"\nNotebook execution complete.")
print(f"All computations used synthetic {t1_volume.shape} volumes.")
print(f"Utility functions imported from src/image_utils.py: 16 functions used.")